In [ ]:
# Cost center classification
import pandas as pd
from pathlib import Path
# Load data


attendance = pd.read_excel(
    "output/second_reports/attendance_feature_engineered.xlsx"
)

calendar = pd.read_csv(
    "output/dim_calendar.csv"
)

# ==========================================
# 2. STANDARDIZE DATE
# ==========================================

attendance["attendance_date"] = pd.to_datetime(
    attendance["attendance_date"]
).dt.normalize()

calendar["attendance_date"] = pd.to_datetime(
    calendar["attendance_date"]
).dt.normalize()

# ==========================================
# 3. INTEGRATE CALENDAR
# ==========================================

attendance = attendance.merge(
    calendar,
    on="attendance_date",
    how="left",
    validate="many_to_one"
)

# ==========================================
# 4. CHECK INTEGRATION
# ==========================================

print("Attendance rows:", len(attendance))
print(
    "Calendar date matched:",
    attendance["year"].notna().sum()
)

print(
    "Calendar date unmatched:",
    attendance["year"].isna().sum()
)

Attendance rows: 27711
Calendar date matched: 27711
Calendar date unmatched: 0


In [5]:
import pandas as pd
from pathlib import Path

# ==========================================
# 1. CONFIGURATION
# ==========================================



OUTPUT_FILE = "output/attendance_feature_engineered.parquet"



# ==========================================
# 3. CHECK REQUIRED COLUMNS
# ==========================================

required_columns = [
    "employee_id",
    "attendance_date",
    "cost_center",
    "is_national_holiday",
    "has_clock_in",
    "has_clock_out",
    "has_attendance_activity"
]

missing_columns = [
    col for col in required_columns
    if col not in attendance.columns
]

if missing_columns:
    raise ValueError(
        f"Missing columns: {missing_columns}"
    )

# ==========================================
# 4. STANDARDIZE DATA TYPES
# ==========================================

attendance["attendance_date"] = pd.to_datetime(
    attendance["attendance_date"]
).dt.normalize()

attendance["cost_center"] = (
    attendance["cost_center"]
    .astype("string")
    .str.strip()
)

# ==========================================
# 5. COST CENTER BUSINESS RULE
# ==========================================

is_fixed_5 = attendance["cost_center"].str.startswith(
    "9",
    na=False
)

attendance["cost_center_type"] = "NON_FIXED_6"

attendance.loc[
    is_fixed_5,
    "cost_center_type"
] = "FIXED_5"

attendance["weekly_work_quota"] = (
    attendance["cost_center_type"]
    .map({
        "FIXED_5": 5,
        "NON_FIXED_6": 6
    })
    .astype("int64")
)

# ==========================================
# 6. DAILY WORK REQUIREMENT
# ==========================================

attendance["day_of_week"] = (
    attendance["attendance_date"].dt.dayofweek
)

is_weekday = attendance["day_of_week"].between(0, 4)

is_holiday = (
    attendance["is_national_holiday"]
    .fillna(False)
    .astype(bool)
)

# Default NULL for NON_FIXED_6
attendance["is_expected_workday"] = pd.Series(
    pd.NA,
    index=attendance.index,
    dtype="boolean"
)

# FIXED_5: Monday-Friday, excluding national holidays
attendance.loc[
    is_fixed_5,
    "is_expected_workday"
] = (
    is_weekday & ~is_holiday
)[is_fixed_5]

# Daily requirement
attendance["daily_work_requirement"] = pd.Series(
    pd.NA,
    index=attendance.index,
    dtype="Int64"
)

# Required to work
attendance.loc[
    is_fixed_5 & is_weekday & ~is_holiday,
    "daily_work_requirement"
] = 1

# Not required to work
attendance.loc[
    is_fixed_5 & ~(is_weekday & ~is_holiday),
    "daily_work_requirement"
] = 0

# ==========================================
# 7. WEEKLY ATTRIBUTES
# ==========================================

# Monday as the beginning of the week
attendance["week_start"] = (
    attendance["attendance_date"]
    - pd.to_timedelta(
        attendance["attendance_date"].dt.dayofweek,
        unit="D"
    )
)

# ==========================================
# 8. COUNT NATIONAL HOLIDAYS PER WEEK
# ==========================================

# Create calendar reference from existing attendance data
calendar = (
    attendance[
        ["attendance_date", "is_national_holiday"]
    ]
    .drop_duplicates(subset=["attendance_date"])
    .copy()
)

calendar["day_of_week"] = (
    calendar["attendance_date"].dt.dayofweek
)

calendar["week_start"] = (
    calendar["attendance_date"]
    - pd.to_timedelta(
        calendar["day_of_week"],
        unit="D"
    )
)

calendar["is_national_holiday"] = (
    calendar["is_national_holiday"]
    .fillna(False)
    .astype(bool)
)

weekly_holidays = (
    calendar[
        calendar["is_national_holiday"]
        & calendar["day_of_week"].between(0, 4)
    ]
    .groupby("week_start")
    .size()
    .rename("weekday_national_holidays")
    .reset_index()
)

attendance = attendance.merge(
    weekly_holidays,
    on="week_start",
    how="left",
    validate="many_to_one"
)

attendance["weekday_national_holidays"] = (
    attendance["weekday_national_holidays"]
    .fillna(0)
    .astype("int64")
)

# ==========================================
# 9. WEEKLY REQUIRED DAYS
# ==========================================

attendance["weekly_required_days"] = (
    attendance["weekly_work_quota"]
)

fixed_mask = attendance["cost_center_type"] == "FIXED_5"

attendance.loc[
    fixed_mask,
    "weekly_required_days"
] = (
    5 - attendance.loc[
        fixed_mask,
        "weekday_national_holidays"
    ]
)

# ==========================================
# 10. EXPORT
# ==========================================

Path("output").mkdir(
    parents=True,
    exist_ok=True
)

attendance.to_parquet(
    OUTPUT_FILE,
    index=False
)

print("Feature engineering completed!")
print("Total rows:", len(attendance))

print("\nCost center distribution:")
print(attendance["cost_center_type"].value_counts())

print("\nNew columns:")
print([
    "cost_center_type",
    "weekly_work_quota",
    "day_of_week",
    "is_expected_workday",
    "daily_work_requirement",
    "week_start",
    "weekday_national_holidays",
    "weekly_required_days"
])

print("\nPreview:")
print(attendance.head())

ValueError: Missing columns: ['cost_center', 'has_clock_in', 'has_clock_out', 'has_attendance_activity']